# **Task1: Misspecified simulation with ridge (set $c = 10$, dense $β^{∗}$).**

## I. Environment setup and imports.
Import required tool functions from `/src` and other packages for graphics, initialize the simulation configurations, then create a folder for saving the results.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# import all necessary functions and default configurations from /src
from src import (
    generate_dgp_data, get_observed_block, fit_ridge, eval_oos_performance,
    DGP_DEFAULTS, CQ_GRID, Z_GRID, SEED, 
    FIG_DIR, DPI, FIG_FORMAT
)

# ensure the image saving path exists
os.makedirs(FIG_DIR, exist_ok=True)

## II. Monte Carlo simulations.
This section performs Monte Carlo simulations (e.g., $N=20$ runs) across varying levels of observed complexity ($c \cdot q$) and shrinkage penalty ($z$).

Averaging across multiple runs reduces finite-sample noise and recovers the smooth asymptotic curves predicted by KMZ.

In [ ]:
n_runs = 20  # Number of Monte Carlo repetitions (from 20 to 50)
T_tr = DGP_DEFAULTS['T_train']
c_dgp = DGP_DEFAULTS['c']
P_total = int(c_dgp * T_tr)

# Data containers for storing metrics across Monte Carlo runs
# Dimensions: (n_runs, len(CQ_GRID), len(Z_GRID))
metrics_names = ['r2_paper', 'exp_return', 'sharpe', 'beta_norm_sq']
results = {m: np.zeros((n_runs, len(CQ_GRID), len(Z_GRID))) for m in metrics_names}

print("Running Monte Carlo Simulations for Task 1...")

for run in tqdm(range(n_runs)):
    # Generate datasets and apply random permutations
    seed = SEED + run
    S_tr, R_tr, S_te, R_te, beta_star, P = generate_dgp_data(**DGP_DEFAULTS, seed=seed)
    rng = np.random.default_rng(seed)
    permuted_indices = rng.permutation(P_total)
    
    # Iterate over the observed complexity grid (cq)
    for i, cq in enumerate(CQ_GRID):
        P1 = int(round(cq * T_tr))
        X_tr, X_te = get_observed_block(S_tr, S_te, P1, permuted_indices)
        
        # Iterate over the shrinkage penalty grid (z)
        for j, z in enumerate(Z_GRID):
            beta_hat = fit_ridge(X_tr, R_tr, z)
            metrics = eval_oos_performance(beta_hat, X_te, R_te)
            for m in metrics_names:
                results[m][run, i, j] = metrics[m]

# Compute Monte Carlo expectations (average across runs)
avg_results = {m: np.mean(results[m], axis=0) for m in metrics_names}

print("Simulation finished successfully!")

## III. Visualize and export OOS performance curves.
We plot the four key evaluation metrics against observed complexity ($c \cdot q$) for representative shrinkage levels ($z$). The vertical dashed line at $c \cdot q=1$ indicates the interpolation threshold where $P_1 = T_{train}$.

In [ ]:
# Select a representative subset of z values for clean plotting
plot_z_indices = [0, 1, 3, 5, 7, 9]  # Corresponds to z = 0, 0.1, 0.5, 2, 10, 50

# Figure 1: OOS R2_paper
plt.figure(figsize=(8, 5), dpi=DPI)
for j in plot_z_indices:
    z_val = Z_GRID[j]
    label = f"z = {z_val} (Ridgeless)" if z_val == 0 else f"z = {z_val}"
    plt.plot(CQ_GRID, avg_results['r2_paper'][:, j], marker='o', label=label)

plt.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='cq = 1 (Interpolation Peak)')
plt.title("OOS $R^2_{paper}$ vs. Observed Complexity ($cq$)", fontsize=12)
plt.xlabel("Observed Complexity ($cq = P_1 / T_{tr}$)", fontsize=10)
plt.ylabel("OOS $R^2_{paper}$", fontsize=10)
plt.ylim(-2.0, 1.0)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/task1_r2_paper.{FIG_FORMAT}", dpi=DPI)
plt.show()

In [ ]:
# Figure 2: Expected Timing Return
plt.figure(figsize=(8, 5), dpi=DPI)
for j in plot_z_indices:
    plt.plot(CQ_GRID, avg_results['exp_return'][:, j], marker='o', label=f"z = {Z_GRID[j]}")

plt.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='cq = 1')
plt.title("Expected Timing Return $\mathbb{E}[R^{\pi}]$ vs. Observed Complexity ($cq$)", fontsize=12)
plt.xlabel("Observed Complexity ($cq$)", fontsize=10)
plt.ylabel("Expected Return $\mathbb{E}[R^{\pi}]$", fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/task1_expected_return.{FIG_FORMAT}", dpi=DPI)
plt.show()

In [ ]:
# Figure 3: Sharpe Ratio
plt.figure(figsize=(8, 5), dpi=DPI)
for j in plot_z_indices:
    plt.plot(CQ_GRID, avg_results['sharpe'][:, j], marker='o', label=f"z = {Z_GRID[j]}")

plt.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='cq = 1')
plt.title("Sharpe Ratio vs. Observed Complexity ($cq$)", fontsize=12)
plt.xlabel("Observed Complexity ($cq$)", fontsize=10)
plt.ylabel("Sharpe Ratio", fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/task1_sharpe_ratio.{FIG_FORMAT}", dpi=DPI)
plt.show()

In [ ]:
# Figure 4: Parameter Norm Squared ||beta||^2 (Log Scale)
plt.figure(figsize=(8, 5), dpi=DPI)
for j in plot_z_indices:
    plt.plot(CQ_GRID, avg_results['beta_norm_sq'][:, j], marker='o', label=f"z = {Z_GRID[j]}")

plt.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='cq = 1')
plt.yscale('log')
plt.title("Parameter Norm $||\hat{\\beta}||_2^2$ vs. Observed Complexity ($cq$) [Log Scale]", fontsize=12)
plt.xlabel("Observed Complexity ($cq$)", fontsize=10)
plt.ylabel("$||\hat{\\beta}||_2^2$ (Log Scale)", fontsize=10)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/task1_beta_norm.{FIG_FORMAT}", dpi=DPI)
plt.show()